# 🚀 Transformer Pulse Deinterleaving - Google Colab Training

Train the Transformer model for radar pulse deinterleaving on Google Colab GPU.

**Paper:** [Radar Pulse Deinterleaving with Transformer Based Deep Metric Learning](https://arxiv.org/abs/2503.13476)

---

## 📋 Before Starting:

1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → **GPU (T4 recommended)**
2. **Mount Google Drive**: Optional but recommended for saving checkpoints
3. **Free GPU limits**: ~12-15 hours/session, may disconnect

---

## ⚙️ Hardware Info:

In [ ]:
# Check GPU availability
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 💾 Mount Google Drive (Recommended)

Save checkpoints to Google Drive to avoid losing progress if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create directories in Drive
!mkdir -p /content/drive/MyDrive/deinterleaving_project/outputs
!mkdir -p /content/drive/MyDrive/deinterleaving_project/data

print("✓ Google Drive mounted successfully!")

## 📦 Installation

Install the challenge package and dependencies.

In [ ]:
%%capture
# Clone repository
!git clone https://github.com/egunn-turing/turing-deinterleaving-challenge.git
%cd turing-deinterleaving-challenge

# Install challenge package
!pip install -e .

# Install model dependencies
%cd models_implementation
!pip install -r requirements.txt

print("✓ Installation complete!")

## 📥 Download Dataset

Download data from HuggingFace. You'll need a token for private datasets.

**Option 1**: Download to Colab /content (faster, deleted after session)  
**Option 2**: Download to Google Drive (persistent, slower)

In [ ]:
# Set your HuggingFace token (if needed)
import os
from getpass import getpass

# Option 1: Enter token manually
HF_TOKEN = getpass("Enter your HuggingFace token (or press Enter to skip): ")
if HF_TOKEN:
    os.environ['HUGGING_FACE_TOKEN'] = HF_TOKEN
    print("✓ Token set")
else:
    print("⚠ No token provided - may hit rate limits")

In [ ]:
import sys
sys.path.insert(0, '/content/turing-deinterleaving-challenge/src')

from turing_deinterleaving_challenge import download_dataset
from pathlib import Path

# Choose download location
USE_DRIVE = True  # Set to False to use local Colab storage

if USE_DRIVE:
    data_dir = Path('/content/drive/MyDrive/deinterleaving_project/data')
    print("📁 Downloading to Google Drive (persistent)...")
else:
    data_dir = Path('/content/data')
    print("📁 Downloading to Colab storage (faster but temporary)...")

# Download training and validation sets (skip test for now)
print("\nDownloading datasets... This may take 10-30 minutes.")
download_dataset(
    save_dir=data_dir,
    subsets=['train', 'validation'],
    hf_token=os.environ.get('HUGGING_FACE_TOKEN'),
    max_workers=3,
)

print("\n✓ Download complete!")
print(f"Data location: {data_dir}")

## 🧪 Quick Test (Optional)

Run a quick test to verify everything works (5-10 minutes).

In [ ]:
# Quick test with 100 samples
%cd /content/turing-deinterleaving-challenge/models_implementation

!python quick_train.py

print("\n✓ Quick test complete! Ready for full training.")

## 🚀 Full Training

Train the Transformer model. Choose configuration based on available time:

| Config | Epochs | Time (T4) | Time (A100) | V-measure |
|--------|--------|-----------|-------------|------------|
| Quick | 3 | ~2-3h | ~1h | ~0.75 |
| Standard | 8 | ~8-12h | ~3-4h | ~0.88 |
| Extended | 12 | ~15-20h | ~6-8h | ~0.89 |

⚠️ **Colab Free Tier**: Max ~12-15 hours/session, may disconnect randomly

In [ ]:
# Training configuration
TRAINING_CONFIG = "standard"  # Options: "quick", "standard", "extended"

configs = {
    "quick": {
        "num_epochs": 3,
        "batch_size": 8,
        "window_length": 1000,
        "validate_every": 1,
    },
    "standard": {
        "num_epochs": 8,
        "batch_size": 8,
        "window_length": 1000,
        "validate_every": 2,
    },
    "extended": {
        "num_epochs": 12,
        "batch_size": 8,
        "window_length": 1000,
        "validate_every": 2,
    },
}

config = configs[TRAINING_CONFIG]
print(f"📋 Training configuration: {TRAINING_CONFIG}")
print(f"   Epochs: {config['num_epochs']}")
print(f"   Batch size: {config['batch_size']}")
print(f"   Window length: {config['window_length']}")

In [ ]:
# Start training
%cd /content/turing-deinterleaving-challenge/models_implementation

# Set output directory (use Drive if mounted)
if USE_DRIVE:
    output_dir = "/content/drive/MyDrive/deinterleaving_project/outputs"
else:
    output_dir = "/content/outputs"

# Build training command
train_cmd = f"""
python train.py \
    --data_dir {data_dir} \
    --output_dir {output_dir} \
    --batch_size {config['batch_size']} \
    --num_epochs {config['num_epochs']} \
    --learning_rate 0.0001 \
    --window_length {config['window_length']} \
    --min_emitters 2 \
    --validate_every {config['validate_every']} \
    --save_every 1 \
    --num_workers 2
"""

print("🚀 Starting training...\n")
print(f"Command: {train_cmd}\n")
print("⏰ This will take several hours. You can monitor progress below.\n")
print("💡 Tip: Keep this tab open to prevent disconnection.\n")
print("="*60)

!{train_cmd}

## 📊 Monitor Training (Optional)

Launch TensorBoard to monitor training progress in real-time.

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Find the latest run directory
import glob
runs = sorted(glob.glob(f"{output_dir}/run_*/tensorboard"))
if runs:
    latest_run = runs[-1]
    print(f"📊 Launching TensorBoard for: {latest_run}")
    %tensorboard --logdir {latest_run}
else:
    print("⚠ No training runs found yet. Start training first.")

## 🔄 Resume Training (If Disconnected)

If Colab disconnects, use this cell to resume from the last checkpoint.

In [ ]:
# Find the latest checkpoint
import glob
from pathlib import Path

output_dir = "/content/drive/MyDrive/deinterleaving_project/outputs"
runs = sorted(glob.glob(f"{output_dir}/run_*"))

if runs:
    latest_run = runs[-1]
    checkpoints = sorted(glob.glob(f"{latest_run}/checkpoint_epoch_*.pt"))
    
    if checkpoints:
        latest_checkpoint = checkpoints[-1]
        print(f"📂 Found checkpoint: {latest_checkpoint}")
        print("\n⚠ Note: Current train.py doesn't support resume.")
        print("   You can manually load checkpoint and continue training.")
        print("   Or restart training - checkpoints are saved every epoch.")
    else:
        print("⚠ No checkpoints found in latest run")
else:
    print("⚠ No training runs found")

## 📈 Evaluate Trained Model

Evaluate the best model on validation set.

In [ ]:
%cd /content/turing-deinterleaving-challenge/models_implementation

# Find best model
import glob
runs = sorted(glob.glob(f"{output_dir}/run_*"))
if runs:
    latest_run = runs[-1]
    best_model = f"{latest_run}/best_model.pt"
    
    if Path(best_model).exists():
        print(f"📊 Evaluating: {best_model}\n")
        
        !python inference.py \
            --checkpoint {best_model} \
            --data_dir {data_dir} \
            --subset validation \
            --batch_size 8 \
            --save_results {latest_run}/validation_results.json
    else:
        print("⚠ Best model not found. Training may not be complete.")
else:
    print("⚠ No training runs found")

## 💾 Download Trained Model

Download the best model to your local machine.

In [ ]:
from google.colab import files
import glob
from pathlib import Path

# Find best model
runs = sorted(glob.glob(f"{output_dir}/run_*"))
if runs:
    latest_run = runs[-1]
    best_model = f"{latest_run}/best_model.pt"
    
    if Path(best_model).exists():
        print(f"📥 Downloading: {best_model}")
        files.download(best_model)
        print("✓ Download complete!")
    else:
        print("⚠ Best model not found")
else:
    print("⚠ No training runs found")

## 💡 Tips & Troubleshooting

### Colab Limitations:
- **Free tier**: ~12-15 hours GPU/session, may disconnect randomly
- **Pro tier**: ~24 hours, more reliable
- **Pro+**: ~50 hours, best for long training

### Preventing Disconnections:
```javascript
// Run this in browser console to prevent auto-disconnect:
function ClickConnect(){
    console.log("Keeping connection alive");
    document.querySelector("colab-connect-button").shadowRoot.querySelector("#connect").click();
}
setInterval(ClickConnect, 60000);
```

### If Training is Too Slow:
1. Reduce `--batch_size` to 4
2. Reduce `--window_length` to 500
3. Filter data: `--max_emitters 50`
4. Train fewer epochs

### Out of Memory:
```python
# Use smaller batch size
config['batch_size'] = 4
config['window_length'] = 500
```

### Save Checkpoints to Drive:
- Always set `USE_DRIVE = True`
- Checkpoints saved every epoch
- Can resume if disconnected

---

## 📞 Support

- **Issues**: [GitHub](https://github.com/egunn-turing/turing-deinterleaving-challenge/issues)
- **Paper**: [arXiv:2503.13476](https://arxiv.org/abs/2503.13476)
- **Dataset Paper**: [arXiv:2602.03856](https://arxiv.org/abs/2602.03856)

---

**Happy Training! 🚀**